In [29]:
# FIFA World Cup 2026 Prediction Engine
## Notebook 1: Data Audit and Feature Construction

# This notebook:
# - Loads all raw datasets
# - Audits data quality
# - Standardizes team names
# - Creates the historical training dataset
# - Creates the 2026 team feature dataset

In [1]:
import pandas as pd
import numpy as np
import os

In [2]:
# Base folders
RAW_DIR = "data/raw"
PROCESSED_DIR = "data/processed"

file_paths = {
    "matches":            f"{RAW_DIR}/historical/matches_1930_2022.csv",
    "fifa_ranking_2022":  f"{RAW_DIR}/historical/fifa_ranking_2022.csv",
    "pastwinners":        f"{RAW_DIR}/historical/worldcup_pastwinners.csv",
    "performance_history": f"{RAW_DIR}/historical/worldcup_team_performance_history.csv",
    "players_master":     f"{RAW_DIR}/players/players_master.csv",
    "fifa_ranking_2026":  f"{RAW_DIR}/teams/fifa_ranking_2026.csv",
    "schedule_2026":      f"{RAW_DIR}/teams/schedule_2026.csv",
    "team_summary":       f"{RAW_DIR}/teams/team_summary.csv",
    "manager_master":     f"{RAW_DIR}/managers/manager_master.csv",
    "venue_master":       f"{RAW_DIR}/venues/venue_master.csv",
}

In [30]:
## Load Raw Datasets

In [3]:
dataframes = {}

for name, path in file_paths.items():
    try:
        dataframes[name] = pd.read_csv(path)
        print(f"✅ Loaded '{name}': {dataframes[name].shape}")
    except Exception as e:
        print(f"❌ Failed loading '{name}'")
        print(f"   Path: {path}")
        print(f"   Error: {e}")
        break

✅ Loaded 'matches': (964, 44)
✅ Loaded 'fifa_ranking_2022': (211, 7)
✅ Loaded 'pastwinners': (22, 9)
✅ Loaded 'performance_history': (192, 24)
✅ Loaded 'players_master': (1248, 10)
✅ Loaded 'fifa_ranking_2026': (211, 8)
✅ Loaded 'schedule_2026': (72, 10)
✅ Loaded 'team_summary': (48, 12)
✅ Loaded 'manager_master': (48, 6)
✅ Loaded 'venue_master': (16, 8)


In [4]:
file_path = "data/raw/teams/schedule_2026.csv"

for enc in ["utf-8", "utf-8-sig", "latin1", "cp1252"]:
    try:
        df = pd.read_csv(file_path, encoding=enc)
        print(f"✅ SUCCESS with {enc}")
        print(df.shape)
        break
    except Exception as e:
        print(f"❌ FAILED with {enc}")
        print(e)
        print()

✅ SUCCESS with utf-8
(72, 10)


In [5]:
import pandas as pd

file_path = "data/raw/teams/schedule_2026.csv"

with open(file_path, "rb") as f:
    print(f.read(200))

b'Round,Day,Date,Time,Score,Referee,Notes,Year,home_team,away_team\nGroup stage,Thu,11/06/26,13:00 (22:00),2-0,,,2026,Mexico,South Africa\nGroup stage,Thu,11/06/26,20:00 (05:00),2-1,,,2026,Korea Republic,'


In [6]:
df = pd.read_csv("data/raw/teams/schedule_2026.csv", encoding="latin1")
df.to_csv("data/raw/teams/schedule_2026.csv", index=False, encoding="utf-8")

In [7]:
dataframes = {}

for name, path in file_paths.items():
    dataframes[name] = pd.read_csv(path)
    print(f"Loaded '{name}': {dataframes[name].shape}")

Loaded 'matches': (964, 44)
Loaded 'fifa_ranking_2022': (211, 7)
Loaded 'pastwinners': (22, 9)
Loaded 'performance_history': (192, 24)
Loaded 'players_master': (1248, 10)
Loaded 'fifa_ranking_2026': (211, 8)
Loaded 'schedule_2026': (72, 10)
Loaded 'team_summary': (48, 12)
Loaded 'manager_master': (48, 6)
Loaded 'venue_master': (16, 8)


In [31]:
## Data Quality Audit

In [8]:
def inventory_report(df, name):
    """
    Prints a quick summary of a DataFrame:
    - shape (rows, columns)
    - column names and their data types
    - how many missing (null) values each column has
    """
    print("=" * 60)
    print(f"DATASET: {name}")
    print("=" * 60)
    print(f"Shape: {df.shape[0]} rows x {df.shape[1]} columns\n")

    print("Columns and data types:")
    print(df.dtypes)
    print()

    print("Missing values per column:")
    missing = df.isnull().sum()
    # Only show columns that actually have missing values
    missing = missing[missing > 0]
    if missing.empty:
        print("  (none)")
    else:
        print(missing)
    print()

In [9]:
for name, df in dataframes.items():
    inventory_report(df, name)

DATASET: matches
Shape: 964 rows x 44 columns

Columns and data types:
home_team                           object
away_team                           object
home_score                           int64
home_xg                            float64
home_penalty                       float64
away_score                           int64
away_xg                            float64
away_penalty                       float64
home_manager                        object
home_captain                        object
away_manager                        object
away_captain                        object
Attendance                           int64
Venue                               object
Officials                           object
Round                               object
Date                                object
Score                               object
Referee                             object
Notes                               object
Host                                object
Year                      

In [10]:
print("MATCHES sample:")
display(dataframes["matches"].head())

print("\nPERFORMANCE HISTORY sample:")
display(dataframes["performance_history"].head())

print("\nPLAYERS MASTER sample:")
display(dataframes["players_master"].head())

print("\nSCHEDULE 2026 sample:")
display(dataframes["schedule_2026"].head())

MATCHES sample:


,home_team,away_team,home_score,home_xg,home_penalty,away_score,away_xg,away_penalty,home_manager,home_captain,...,home_penalty_shootout_miss_long,away_penalty_shootout_miss_long,home_red_card,away_red_card,home_yellow_red_card,away_yellow_red_card,home_yellow_card_long,away_yellow_card_long,home_substitute_in_long,away_substitute_in_long
0,Argentina,France,3,3.3,4.0,3,2.2,2.0,Lionel Scaloni,Lionel Messi,...,NaN,"['3|1:1|Kingsley Coman', '5|2:1|Aurélien Tchou...",NaN,NaN,NaN,NaN,"['45+7&rsquor;|2:0|Enzo Fernández', '90+8&rsqu...","['55&rsquor;|2:0|Adrien Rabiot', '87&rsquor;|2...",['64&rsquor;|2:0|Marcos Acuña|for Ángel Di Mar...,['41&rsquor;|2:0|Randal Kolo Muani|for Ousmane...
1,Croatia,Morocco,2,0.7,NaN,1,1.2,NaN,Zlatko Dalić,Luka Modrić,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"['69&rsquor;|2:1|Azzedine Ounahi', '84&rsquor;...",['61&rsquor;|2:1|Nikola Vlašić|for Andrej Kram...,['46&rsquor;|2:1|Ilias Chair|for Abdelhamid Sa...
2,France,Morocco,2,2.0,NaN,0,0.9,NaN,Didier Deschamps,Hugo Lloris,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,['27&rsquor;|1:0|Sofiane Boufal'],['65&rsquor;|1:0|Marcus Thuram|for Olivier Gir...,['21&rsquor;|1:0|Selim Amallah|for Romain Saïs...
3,Argentina,Croatia,3,2.3,NaN,0,0.5,NaN,Lionel Scaloni,Lionel Messi,...,NaN,NaN,NaN,NaN,NaN,NaN,"['68&rsquor;|2:0|Cristian Romero', '71&rsquor;...","['32&rsquor;|0:0|Mateo Kovačić', '32&rsquor;|0...",['62&rsquor;|2:0|Lisandro Martínez|for Leandro...,"['46&rsquor;|2:0|Mislav Oršić|for Borna Sosa',..."
4,Morocco,Portugal,1,1.4,NaN,0,0.9,NaN,Hoalid Regragui,Romain Saïss,...,NaN,NaN,NaN,NaN,Walid Cheddira · 90+3,NaN,"['70&rsquor;|1:0|Achraf Dari', '90+1&rsquor;|1...",['87&rsquor;|1:0|Vitinha'],['57&rsquor;|1:0|Achraf Dari|for Romain Saïss'...,['51&rsquor;|1:0|João Cancelo|for Raphaël Guer...



PERFORMANCE HISTORY sample:


,version,team,continent,is_host,goals_scored_last_4y,goals_received_last_4y,wins_last_4y,losses_last_4y,draws_last_4y,world_cup_titles_before,...,world_cup_participations_before,groups_passed_before,round16_before,quarterfinals_before,semifinals_before,finals_before,winner,finalist,semi_finalist,quarter_finalist
0,2006,Angola,Africa,0,61,49,19,13,14,0,...,0,0,0,0,0,0,0,0,0,0
1,2006,Argentina,South America,0,97,55,31,10,10,2,...,13,10,5,6,4,4,0,0,0,1
2,2006,Australia,Oceania,0,101,34,23,8,5,0,...,1,0,0,0,0,0,0,0,0,0
3,2006,Brazil,South America,0,117,47,30,9,17,5,...,17,15,7,11,9,6,0,0,0,1
4,2006,Costa Rica,North America,0,89,84,26,25,11,0,...,2,1,1,0,0,0,0,0,0,0



PLAYERS MASTER sample:


,team,shirt_number,player_name,position,date_of_birth,club,height_cm,international_caps,international_goals,coach
0,Algeria,1,MASTIL Melvin,GK,19/02/2000,FC Stade Nyonnais (SUI),194,2,0,PETKOVIC Vladimir
1,Algeria,2,MANDI Aissa,DF,22/10/1991,Lille OSC (FRA),184,119,8,PETKOVIC Vladimir
2,Algeria,3,ABADA Achref,DF,15/06/1999,USM Alger (ALG),185,10,1,PETKOVIC Vladimir
3,Algeria,4,TOUGAI Mohamed Amine,DF,22/01/2000,Espérance De Tunisie (TUN),186,30,2,PETKOVIC Vladimir
4,Algeria,5,BELAID Zineddine,DF,20/03/1999,JS Kabylie (ALG),186,18,1,PETKOVIC Vladimir



SCHEDULE 2026 sample:


,Round,Day,Date,Time,Score,Referee,Notes,Year,home_team,away_team
0,Group stage,Thu,11/06/26,13:00 (22:00),2-0,NaN,NaN,2026,Mexico,South Africa
1,Group stage,Thu,11/06/26,20:00 (05:00),2-1,NaN,NaN,2026,Korea Republic,Czechia
2,Group stage,Fri,12/06/26,15:00 (22:00),1-1,NaN,NaN,2026,Canada,Bosnia-Herzegovina
3,Group stage,Fri,12/06/26,18:00 (04:00),4-1,NaN,NaN,2026,United States,Paraguay
4,Group stage,Sat,13/06/26,12:00 (22:00),1-1,NaN,NaN,2026,Qatar,Switzerland


In [11]:
players = dataframes["players_master"]

# Total number of player rows
print("Total players:", len(players))

# How many unique teams appear?
print("Unique teams:", players["team"].nunique())

# Count players per team
players_per_team = players.groupby("team").size()

print("\nTeams with player count != 26:")
problem_teams = players_per_team[players_per_team != 26]
if problem_teams.empty:
    print("  (none - every team has exactly 26 players)")
else:
    print(problem_teams)

Total players: 1248
Unique teams: 48

Teams with player count != 26:
  (none - every team has exactly 26 players)


In [32]:
## Team Name Standardization

In [12]:
team_name_sources = {
    "matches_home":        (dataframes["matches"], "home_team"),
    "matches_away":        (dataframes["matches"], "away_team"),
    "performance_history": (dataframes["performance_history"], "team"),
    "players_master":      (dataframes["players_master"], "team"),
    "fifa_ranking_2026":   (dataframes["fifa_ranking_2026"], "team"),
    "team_summary":        (dataframes["team_summary"], "team"),
    "manager_master":      (dataframes["manager_master"], "country"),
    "schedule_home":       (dataframes["schedule_2026"], "home_team"),
    "schedule_away":       (dataframes["schedule_2026"], "away_team"),
}

unique_names = {}

for label, (df, col) in team_name_sources.items():
    unique_names[label] = set(df[col].dropna().unique())
    print(f"{label}: {len(unique_names[label])} unique values")

matches_home: 82 unique values
matches_away: 86 unique values
performance_history: 62 unique values
players_master: 48 unique values
fifa_ranking_2026: 211 unique values
team_summary: 48 unique values
manager_master: 48 unique values
schedule_home: 48 unique values
schedule_away: 48 unique values


In [13]:
canonical_2026 = unique_names["players_master"]

datasets_to_check = [
    "fifa_ranking_2026",
    "team_summary",
    "manager_master",
    "schedule_home",
    "schedule_away",
]

for label in datasets_to_check:
    other = unique_names[label]

    missing_from_other = canonical_2026 - other   
    extra_in_other = other - canonical_2026       

    print(f"--- {label} ---")
    print("  In players_master but missing here:", missing_from_other if missing_from_other else "(none)")
    print("  Here but not in players_master:    ", extra_in_other if extra_in_other else "(none)")
    print()

--- fifa_ranking_2026 ---
  In players_master but missing here: {'Bosnia And Herzegovina', "Côte D'Ivoire"}
  Here but not in players_master:     {'Zambia', 'Comoros', 'Italy', 'Kyrgyz Republic', 'Sudan', 'San Marino', 'Puerto Rico', 'El Salvador', 'Sierra Leone', 'Bahrain', 'Tanzania', 'Singapore', 'Bangladesh', 'Cuba', 'Chinese Taipei', 'Turks and Caicos Islands', 'Venezuela', 'Nicaragua', 'Dominican Republic', 'Nepal', 'Grenada', 'Eritrea', 'Zimbabwe', 'Vanuatu', 'Lithuania', 'British Virgin Islands', 'Kosovo', 'Yemen', 'United Arab Emirates', 'Macau', 'Congo', 'Eswatini', 'St Lucia', 'Madagascar', 'Botswana', 'Pakistan', 'Burkina Faso', 'Peru', 'Iceland', 'Poland', 'China PR', 'St Kitts and Nevis', 'Turkmenistan', 'Slovakia', 'Thailand', 'Belize', 'Georgia', 'Finland', 'Slovenia', 'Oman', 'Trinidad and Tobago', 'Djibouti', 'Estonia', 'Cambodia', 'Barbados', 'Chad', 'Samoa', 'Romania', 'Myanmar', 'Albania', 'Mauritius', 'Antigua and Barbuda', 'Mali', 'São Tomé and Príncipe', 'The Ga

In [14]:
historical_datasets = ["matches_home", "matches_away", "performance_history"]

for label in historical_datasets:
    other = unique_names[label]

    overlap = canonical_2026 & other            # names that match exactly
    not_in_history = canonical_2026 - other      # 2026 teams never seen historically
    historical_only = other - canonical_2026     # historical names not in 2026 list

    print(f"--- {label} ---")
    print(f"  Exact matches with 2026 teams: {len(overlap)} / {len(canonical_2026)}")
    print(f"  2026 teams never appearing here: {not_in_history if not_in_history else '(none)'}")
    print(f"  Names here not matching any 2026 team ({len(historical_only)} total):")
    print(f"    {sorted(historical_only)}")
    print()

--- matches_home ---
  Exact matches with 2026 teams: 39 / 48
  2026 teams never appearing here: {'Jordan', 'Bosnia And Herzegovina', 'Curaçao', 'USA', 'Cabo Verde', 'Uzbekistan', 'Congo DR', 'Czechia', "Côte D'Ivoire"}
  Names here not matching any 2026 team (43 total):
    ['Angola', 'Bolivia', 'Bosnia and Herzegovina', 'Bulgaria', 'Cameroon', 'Chile', 'China PR', 'Costa Rica', 'Cuba', 'Czech Republic', 'Czechoslovakia', "Côte d'Ivoire", 'Denmark', 'FR Yugoslavia', 'Germany DR', 'Greece', 'Honduras', 'Hungary', 'Iceland', 'Italy', 'Jamaica', 'Korea DPR', 'Nigeria', 'Northern Ireland', 'Peru', 'Poland', 'Republic of Ireland', 'Romania', 'Russia', 'Serbia', 'Serbia and Montenegro', 'Slovakia', 'Slovenia', 'Soviet Union', 'Togo', 'Trinidad and Tobago', 'Ukraine', 'United Arab Emirates', 'United States', 'Wales', 'West Germany', 'Yugoslavia', 'Zaire']

--- matches_away ---
  Exact matches with 2026 teams: 39 / 48
  2026 teams never appearing here: {'Jordan', 'Bosnia And Herzegovina', 'Cu

In [15]:
team_name_mapping = {

    # USA
    "United States": "USA",

    # Korea
    "South Korea": "Korea Republic",
    "Korea, South": "Korea Republic",
    "Korea DPR": "North Korea",

    # Iran
    "Iran": "IR Iran",

    # Ivory Coast
    "Ivory Coast": "Côte D'Ivoire",
    "Côte d'Ivoire": "Côte D'Ivoire",
    "Cote d'Ivoire": "Côte D'Ivoire",
    "Cote d Ivoire": "Côte D'Ivoire",
    "C\x99te d'Ivoire": "Côte D'Ivoire",

    # DR Congo
    "DR Congo": "Congo DR",

    # Cape Verde
    "Cape Verde": "Cabo Verde",

    # Bosnia
    "Bosnia and Herzegovina": "Bosnia And Herzegovina",
    "Bosnia-Herzegovina": "Bosnia And Herzegovina",

    # Curacao
    "Curacao": "Curaçao",
    "Cura\x8dao": "Curaçao",

    # Turkey
    "Turkey": "Türkiye",
    "T\x9frkiye": "Türkiye",

    # Czechia
    "Czech Republic": "Czechia"
}

print(f"Mapping contains {len(team_name_mapping)} entries")

Mapping contains 19 entries


In [16]:
team_name_mapping.update({
    "CÃ\x82Â\x99te d'Ivoire": "Côte D'Ivoire",
    "CuraÃ\x82Â\x8dao": "Curaçao",
    "TÃ\x82Â\x9frkiye": "Türkiye",
})

In [17]:
def standardize_team_names(df, columns, mapping):
    """
    Returns a COPY of df where every value in the given columns
    is replaced according to `mapping`, if a replacement exists.
    Values not found in the mapping are left unchanged.
    """
    df = df.copy()
    for col in columns:
        df[col] = df[col].replace(mapping)
    return df


# Apply standardization to every dataset that contains team names
dataframes["matches"] = standardize_team_names(
    dataframes["matches"], ["home_team", "away_team"], team_name_mapping
)
dataframes["performance_history"] = standardize_team_names(
    dataframes["performance_history"], ["team"], team_name_mapping
)
dataframes["fifa_ranking_2026"] = standardize_team_names(
    dataframes["fifa_ranking_2026"], ["team"], team_name_mapping
)
dataframes["team_summary"] = standardize_team_names(
    dataframes["team_summary"], ["team"], team_name_mapping
)
dataframes["manager_master"] = standardize_team_names(
    dataframes["manager_master"], ["country"], team_name_mapping
)
dataframes["schedule_2026"] = standardize_team_names(
    dataframes["schedule_2026"], ["home_team", "away_team"], team_name_mapping
)

print("Standardization applied to all datasets.")

Standardization applied to all datasets.


In [18]:
# Re-extract unique names now that standardization has been applied
unique_names_after = {}
for label, (_, col) in team_name_sources.items():
    # Note: re-fetch the dataframe from `dataframes` in case it was updated
    df_key = {
        "matches_home": "matches", "matches_away": "matches",
        "performance_history": "performance_history",
        "players_master": "players_master",
        "fifa_ranking_2026": "fifa_ranking_2026",
        "team_summary": "team_summary",
        "manager_master": "manager_master",
        "schedule_home": "schedule_2026", "schedule_away": "schedule_2026",
    }[label]
    unique_names_after[label] = set(dataframes[df_key][col].dropna().unique())

# Re-run the same canonical comparison as Cell 9
for label in ["fifa_ranking_2026", "team_summary", "manager_master", "schedule_home", "schedule_away"]:
    other = unique_names_after[label]
    missing = canonical_2026 - other
    extra = other - canonical_2026
    print(f"--- {label} ---")
    print("  Still missing:", missing if missing else "(none - all matched!)")
    print("  Still extra:  ", extra if extra else "(none - all matched!)")
    print()

--- fifa_ranking_2026 ---
  Still missing: (none - all matched!)
  Still extra:   {'Zambia', 'Comoros', 'Italy', 'Kyrgyz Republic', 'Sudan', 'San Marino', 'Puerto Rico', 'El Salvador', 'Sierra Leone', 'Bahrain', 'Tanzania', 'Singapore', 'Bangladesh', 'Cuba', 'Chinese Taipei', 'Turks and Caicos Islands', 'Venezuela', 'Nicaragua', 'Dominican Republic', 'Nepal', 'Grenada', 'Eritrea', 'Zimbabwe', 'Vanuatu', 'Lithuania', 'British Virgin Islands', 'Kosovo', 'Yemen', 'United Arab Emirates', 'Macau', 'Congo', 'Eswatini', 'St Lucia', 'Madagascar', 'Botswana', 'Pakistan', 'Burkina Faso', 'Peru', 'Iceland', 'Poland', 'China PR', 'St Kitts and Nevis', 'Turkmenistan', 'Slovakia', 'Thailand', 'Belize', 'Georgia', 'Finland', 'Slovenia', 'Oman', 'Trinidad and Tobago', 'Djibouti', 'Estonia', 'Cambodia', 'Barbados', 'Chad', 'Samoa', 'Romania', 'Myanmar', 'Albania', 'Mauritius', 'Antigua and Barbuda', 'Mali', 'São Tomé and Príncipe', 'The Gambia', 'Cameroon', 'Azerbaijan', 'Costa Rica', 'Northern Ireland

In [33]:
## Historical Feature Construction

In [19]:
matches = dataframes["matches"].copy()
perf = dataframes["performance_history"].copy()

# Create common merge key
matches["year"] = matches["Year"]
perf["year"] = perf["version"]

# Keep only years for which performance_history exists
valid_years = perf["year"].unique()

matches_filtered = matches[
    matches["year"].isin(valid_years)
].copy()

print("Original matches:", len(matches))
print("Filtered matches:", len(matches_filtered))


def prefix_feature_columns(df, prefix):
    rename_map = {
        col: f"{prefix}_{col}"
        for col in df.columns
        if col not in ["team", "year"]
    }
    return df.rename(columns=rename_map)


# Home features
perf_home = prefix_feature_columns(perf, "home")
perf_home = perf_home.rename(columns={"team": "home_team"})

# Away features
perf_away = prefix_feature_columns(perf, "away")
perf_away = perf_away.rename(columns={"team": "away_team"})

# Merge home features
training = matches_filtered.merge(
    perf_home,
    on=["home_team", "year"],
    how="left"
)

# Merge away features
training = training.merge(
    perf_away,
    on=["away_team", "year"],
    how="left"
)

print("\nTraining shape:", training.shape)

print("\nMissing home features:")
print(training["home_continent"].isna().sum())

print("\nMissing away features:")
print(training["away_continent"].isna().sum())

Original matches: 964
Filtered matches: 384

Training shape: (384, 91)

Missing home features:
0

Missing away features:
0


In [20]:
HOME_SCORE_COL = "home_score"
AWAY_SCORE_COL = "away_score"


def get_match_result(row):
    if row[HOME_SCORE_COL] > row[AWAY_SCORE_COL]:
        return "Home Win"
    elif row[HOME_SCORE_COL] < row[AWAY_SCORE_COL]:
        return "Away Win"
    else:
        return "Draw"


training["match_result"] = training.apply(get_match_result, axis=1)

print(training["match_result"].value_counts())
print("\nPercentages:")
print(training["match_result"].value_counts(normalize=True).round(3))

match_result
Home Win    165
Away Win    131
Draw         88
Name: count, dtype: int64

Percentages:
match_result
Home Win    0.430
Away Win    0.341
Draw        0.229
Name: proportion, dtype: float64


In [21]:
sample_feature_col = [c for c in training.columns if c.startswith("home_")][0]
sample_feature_col_away = [c for c in training.columns if c.startswith("away_")][0]

home_matched = training[sample_feature_col].notna().sum()
away_matched = training[sample_feature_col_away].notna().sum()
both_matched = (training[sample_feature_col].notna() & training[sample_feature_col_away].notna()).sum()
neither_matched = (training[sample_feature_col].isna() & training[sample_feature_col_away].isna()).sum()

total = len(training)

print(f"Total matches: {total}")
print(f"Home features found: {home_matched} ({home_matched/total:.1%})")
print(f"Away features found: {away_matched} ({away_matched/total:.1%})")
print(f"BOTH found:          {both_matched} ({both_matched/total:.1%})")
print(f"NEITHER found:       {neither_matched} ({neither_matched/total:.1%})")

Total matches: 384
Home features found: 384 (100.0%)
Away features found: 384 (100.0%)
BOTH found:          384 (100.0%)
NEITHER found:       0 (0.0%)


In [22]:
# Find rows where home features are missing
unmatched_home = training[training[sample_feature_col].isna()]
unmatched_away = training[training[sample_feature_col_away].isna()]

print("Unmatched HOME (team, year) combinations:")
print(unmatched_home[["year", "home_team"]].drop_duplicates().sort_values("year"))

print("\nUnmatched AWAY (team, year) combinations:")
print(unmatched_away[["year", "away_team"]].drop_duplicates().sort_values("year"))

Unmatched HOME (team, year) combinations:
Empty DataFrame
Columns: [year, home_team]
Index: []

Unmatched AWAY (team, year) combinations:
Empty DataFrame
Columns: [year, away_team]
Index: []


In [23]:
os.makedirs(PROCESSED_DIR, exist_ok=True)

output_path = f"{PROCESSED_DIR}/training_dataset_raw.csv"
training.to_csv(output_path, index=False)

print(f"Saved {training.shape[0]} rows x {training.shape[1]} columns to:")
print(f"  {output_path}")

Saved 384 rows x 92 columns to:
  data/processed/training_dataset_raw.csv


In [34]:
## 2026 Team Feature Creation

In [24]:
team_summary = dataframes["team_summary"]
fifa_ranking_2026 = dataframes["fifa_ranking_2026"]
manager_master = dataframes["manager_master"]

team_features_2026 = team_summary.merge(
    fifa_ranking_2026, on="team", how="left"
)

team_features_2026 = team_features_2026.merge(
    manager_master, left_on="team", right_on="country", how="left"
)

# Dropping the redundant 'country' column since it duplicates 'team'
team_features_2026 = team_features_2026.drop(columns=["country"])

print("team_features_2026 shape:", team_features_2026.shape)
print("\nColumns:")
print(list(team_features_2026.columns))

team_features_2026 shape: (48, 24)

Columns:
['team', 'squad_size', 'average_age', 'average_height', 'total_caps', 'average_caps', 'total_goals', 'average_goals', 'number_of_goalkeepers', 'number_of_defenders', 'number_of_midfielders', 'number_of_forwards', 'team_code', 'association', 'rank', 'previous_rank', 'points', 'previous_points', 'rated_matches', 'manager_name', 'manager_nationality', 'appointment_year', 'preferred_formation', 'style_description']


In [25]:
print("Number of teams:", len(team_features_2026))
print("Expected: 48\n")

print("Duplicate team names:")
duplicates = team_features_2026[team_features_2026["team"].duplicated()]
print(duplicates if not duplicates.empty else "  (none)")

print("\nMissing values per column:")
missing = team_features_2026.isnull().sum()
missing = missing[missing > 0]
print(missing if not missing.empty else "  (none)")

print("\nTeams with any missing values:")
rows_with_missing = team_features_2026[team_features_2026.isnull().any(axis=1)]
print(rows_with_missing["team"].tolist() if not rows_with_missing.empty else "  (none)")

Number of teams: 48
Expected: 48

Duplicate team names:
  (none)

Missing values per column:
  (none)

Teams with any missing values:
  (none)


In [26]:
output_path_2026 = f"{PROCESSED_DIR}/team_features_2026.csv"
team_features_2026.to_csv(output_path_2026, index=False)

print(f"Saved {team_features_2026.shape[0]} rows x {team_features_2026.shape[1]} columns to:")
print(f"  {output_path_2026}")

Saved 48 rows x 24 columns to:
  data/processed/team_features_2026.csv


In [27]:
print("=" * 60)
print("DATA AUDIT & MERGE — SUMMARY")
print("=" * 60)

print(f"\nHistorical training table:")
print(f"  Rows: {training.shape[0]}  Columns: {training.shape[1]}")
print(f"  Match result distribution: {dict(training['match_result'].value_counts())}")
print(f"  Rows with complete home+away features: {both_matched} / {total} ({both_matched/total:.1%})")

print(f"\n2026 team features table:")
print(f"  Rows: {team_features_2026.shape[0]} (expected 48)")
print(f"  Columns: {team_features_2026.shape[1]}")

print(f"\nTeam name mapping entries applied: {len(team_name_mapping)}")

print(f"\nOutputs saved:")
print(f"  - {PROCESSED_DIR}/training_dataset_raw.csv")
print(f"  - {PROCESSED_DIR}/team_features_2026.csv")

DATA AUDIT & MERGE — SUMMARY

Historical training table:
  Rows: 384  Columns: 92
  Match result distribution: {'Home Win': np.int64(165), 'Away Win': np.int64(131), 'Draw': np.int64(88)}
  Rows with complete home+away features: 384 / 384 (100.0%)

2026 team features table:
  Rows: 48 (expected 48)
  Columns: 24

Team name mapping entries applied: 22

Outputs saved:
  - data/processed/training_dataset_raw.csv
  - data/processed/team_features_2026.csv
